In [1]:
# done with poetry
# %pip install pandas
# %pip install scikit-learn
# %pip install xgboost

In [2]:
import os

import pandas as pd
import numpy as np


from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor

## Data loading

In [3]:
data_folder = "sample_data"

df_bu_feat = pd.read_csv(os.path.join(data_folder,"bu_feat.csv.gz")) 
df_train = pd.read_csv(os.path.join(data_folder, "train.csv.gz"))
df_test = pd.read_csv(os.path.join(data_folder, "test.csv.gz")) 

### Merging features

In [4]:
df_train_feat = pd.merge(df_train, df_bu_feat, how="left", on = "but_num_business_unit")
df_test_feat = pd.merge(df_test, df_bu_feat, how="left", on = "but_num_business_unit")

### Split train, val set

In [5]:
df_train_feat.head(5)

,day_id,but_num_business_unit,dpt_num_department,turnover,but_postcode,but_latitude,but_longitude,but_region_idr_region,zod_idr_zone_dgr
0,2017-09-30,64,127,580.308443,16400,45.625172,0.111939,70,10
1,2017-09-30,119,127,1512.995918,74100,46.195037,6.254448,51,4
2,2017-09-30,4,88,668.593556,6600,43.600994,7.078160,55,10
3,2017-09-30,425,127,0.000000,59000,50.617921,3.084186,33,3
4,2017-09-30,513,73,0.000000,33610,44.717366,-0.733429,33,3


In [6]:
df_train_feat.shape

(277719, 9)

In [7]:
# Train and val set

df_train_feat["day_id"] = pd.to_datetime(df_train_feat["day_id"])
df_train_feat["day_id_week"] = df_train_feat.day_id.dt.isocalendar().week
df_train_feat["day_id_month"] = df_train_feat["day_id"].dt.month
df_train_feat["day_id_year"] = df_train_feat["day_id"].dt.year

# 2017 must a variable
year_split = 2017
df_train = df_train_feat[(df_train_feat.day_id_year < year_split)]
df_val = df_train_feat[(df_train_feat.day_id_year == 2017)]

y_train = df_train.turnover
y_val = df_val.turnover

### Scikit pipeline

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin

In [9]:
# this is a custom transformer -> dependency alert!
class CustomPreprocressing(BaseEstimator, TransformerMixin):
    """
    This class includes all the steps for the preprocessing
    """
    def __init__(self, cat_cols):
        """
        Initialize the class / Can be empty
        """
        self.cat_cols = cat_cols

    def fit(self, X, y=None):
        """
        This method is only created so that the pipeline containing this transformer does not raise an error
        """
        return self

    def transform(self, data):
        """
        Inputs :
          -- data : DataFrame, DataFrame contening all the data needed for the model
        Outputs :
          -- DataFrame, DataFrame prepared for modeling

        """
        data.loc[:,"day_id"] = pd.to_datetime(data["day_id"])
        data.loc[:,"day_id_week"] = data.day_id.dt.isocalendar().week
        data.loc[:,"day_id_month"] = data["day_id"].dt.month
        data.loc[:,"day_id_year"] = data["day_id"].dt.year
        data.loc[:,self.cat_cols] = data[self.cat_cols].apply(lambda x: x.astype(str))
        return data

In [10]:

num_attrib = ["but_latitude","but_longitude", 'day_id_year']
cat_attrib = [
            "day_id_week",
            "day_id_month",
            "but_region_idr_region",
            "zod_idr_zone_dgr",
            "but_num_business_unit",
            "dpt_num_department",
        ]

num_pipeline = Pipeline([
    ('std_scaler', StandardScaler()),
])
cat_onehot_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown="ignore")),
])
preparation_pipeline = ColumnTransformer([
    ("num",num_pipeline, num_attrib),
    ("cat_onehot", cat_onehot_pipeline, cat_attrib)
])

full_pipeline = Pipeline([
    ('preprocessing', CustomPreprocressing(cat_cols=cat_attrib )),
    ('preparation', preparation_pipeline),
    ('model', GradientBoostingRegressor())
])

In [11]:
model_final = full_pipeline.fit(df_train, y_train)
y_predict_val = model_final.predict(df_val)

metric_mae = mean_absolute_error(y_val, y_predict_val)
print(metric_mae)

/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['12' '12' '12' ... '12' '12' '12']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  data.loc[:,self.cat_cols] = data[self.cat_cols].apply(lambda x: x.astype(str))
/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['107' '33' '134' ... '30' '75' '71']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:,self.cat_cols] = data[self.cat_cols].apply(lambda x: x.astype(str))
/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['6' '3' '10' ... '6' '6' '10']' has dtype incompatible with int64, please explicitly cast to a compatible dtype firs

407.22893116868767


/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['9' '9' '9' ... '1' '1' '1']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  data.loc[:,self.cat_cols] = data[self.cat_cols].apply(lambda x: x.astype(str))
/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['70' '51' '55' ... '178' '64' '4']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:,self.cat_cols] = data[self.cat_cols].apply(lambda x: x.astype(str))
/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['10' '4' '10' ... '72' '10' '4']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  d

In [12]:
df_val['dpt_num_department'].unique()

array(['127', '88', '73', '117'], dtype=object)

In [13]:
df_val['prediction'] = y_predict_val
filter_ts = lambda x: (x.but_num_business_unit=="32") & (x.dpt_num_department=='73')
display(df_val[filter_ts].head(10))

display(df_train[filter_ts].head(10))

/tmp/ipykernel_27625/2715622416.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_val['prediction'] = y_predict_val


,day_id,but_num_business_unit,dpt_num_department,turnover,but_postcode,but_latitude,but_longitude,but_region_idr_region,zod_idr_zone_dgr,day_id_week,day_id_month,day_id_year,prediction
1089,2017-09-30,32,73,452.769296,29490,48.425054,-4.436446,7,6,39,9,2017,469.318872
2519,2017-09-23,32,73,513.263277,29490,48.425054,-4.436446,7,6,38,9,2017,469.318872
3432,2017-09-16,32,73,912.694843,29490,48.425054,-4.436446,7,6,37,9,2017,469.318872
4290,2017-09-09,32,73,831.668945,29490,48.425054,-4.436446,7,6,36,9,2017,469.318872
5599,2017-09-02,32,73,528.774473,29490,48.425054,-4.436446,7,6,35,9,2017,469.318872
6357,2017-08-26,32,73,562.253678,29490,48.425054,-4.436446,7,6,34,8,2017,502.036028
7705,2017-08-19,32,73,898.535514,29490,48.425054,-4.436446,7,6,33,8,2017,502.036028
9811,2017-08-12,32,73,1049.221014,29490,48.425054,-4.436446,7,6,32,8,2017,502.036028
10973,2017-08-05,32,73,1082.103732,29490,48.425054,-4.436446,7,6,31,8,2017,502.036028
12303,2017-07-29,32,73,1155.885670,29490,48.425054,-4.436446,7,6,30,7,2017,502.036028


,day_id,but_num_business_unit,dpt_num_department,turnover,but_postcode,but_latitude,but_longitude,but_region_idr_region,zod_idr_zone_dgr,day_id_week,day_id_month,day_id_year
49581,2016-12-31,32,73,544.941047,29490,48.425054,-4.436446,7,6,52,12,2016
50950,2016-12-24,32,73,1023.650512,29490,48.425054,-4.436446,7,6,51,12,2016
51720,2016-12-17,32,73,1231.944726,29490,48.425054,-4.436446,7,6,50,12,2016
53122,2016-12-10,32,73,657.439375,29490,48.425054,-4.436446,7,6,49,12,2016
54834,2016-12-03,32,73,715.383766,29490,48.425054,-4.436446,7,6,48,12,2016
54920,2016-11-26,32,73,650.073868,29490,48.425054,-4.436446,7,6,47,11,2016
56116,2016-11-19,32,73,827.484007,29490,48.425054,-4.436446,7,6,46,11,2016
58308,2016-11-12,32,73,947.222428,29490,48.425054,-4.436446,7,6,45,11,2016
58821,2016-11-05,32,73,976.114374,29490,48.425054,-4.436446,7,6,44,11,2016
59821,2016-10-29,32,73,967.194617,29490,48.425054,-4.436446,7,6,43,10,2016


In [14]:
model_final = full_pipeline.fit(df_train_feat, df_train_feat.turnover.values)


/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['9' '9' '9' ... '12' '12' '12']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  data.loc[:,self.cat_cols] = data[self.cat_cols].apply(lambda x: x.astype(str))
/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['70' '51' '55' ... '30' '75' '71']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:,self.cat_cols] = data[self.cat_cols].apply(lambda x: x.astype(str))
/tmp/ipykernel_27625/2503044936.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['10' '4' '10' ... '6' '6' '10']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
 

In [15]:
y_pred = model_final.predict(df_test_feat)

AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
df_test_feat['prediction'] = y_pred

In [ ]:
display(df_test_feat.head(10))

,day_id,but_num_business_unit,dpt_num_department,but_postcode,but_latitude,but_longitude,but_region_idr_region,zod_idr_zone_dgr,day_id_week,day_id_month,day_id_year,prediction
0,2017-11-25,95,73,80000,49.869382,2.280452,69,4,47,11,2017,28.096508
1,2017-11-25,4,117,6600,43.600994,7.078160,55,10,47,11,2017,324.086069
2,2017-11-25,113,127,84014,43.919562,4.867583,115,10,47,11,2017,1333.614589
3,2017-11-25,93,117,13008,43.239744,5.396694,71,10,47,11,2017,357.205467
4,2017-11-25,66,127,34500,43.347835,3.255024,6,10,47,11,2017,1389.478844
5,2017-11-25,225,88,91220,48.622942,2.301157,2,6,47,11,2017,499.755396
6,2017-11-25,37,117,6210,43.551454,6.948166,55,10,47,11,2017,324.086069
7,2017-11-25,720,73,22100,48.475374,-2.041166,7,6,47,11,2017,169.660511
8,2017-11-25,1015,127,59000,50.636600,3.069400,32,1,47,11,2017,497.073405
9,2017-11-25,505,88,77183,48.829569,2.636586,2,6,47,11,2017,519.015757
